# 11 — Surface-Code UPDE Resource Estimation

Physical qubit budget for surface-code protected Kuramoto-XY simulation.
Structural model — circuit topology, transversal gates, syndrome extraction.

In [ ]:
import matplotlib.pyplot as plt

from scpn_quantum_control.qec.fault_tolerant import RepetitionCodeUPDE
from scpn_quantum_control.qec.surface_code_upde import SurfaceCodeUPDE

## 1. Qubit Budget Comparison

In [ ]:
distances = [3, 5, 7, 9]
osc_counts = [2, 4, 8, 16]

print(f"{'N_osc':>6} {'d':>4} {'Rep.Code':>10} {'Surface':>10} {'Correctable':>12}")
print("-" * 46)
for n_osc in osc_counts:
    for d in distances:
        rep = RepetitionCodeUPDE(n_osc=n_osc, code_distance=d)
        sc = SurfaceCodeUPDE(n_osc=n_osc, code_distance=d)
        budget = sc.physical_qubit_budget()
        print(
            f"{n_osc:6d} {d:4d} {rep.physical_qubit_count():10d} {budget['total_physical']:10d} {budget['correctable_errors']:12d}"
        )

## 2. Scaling Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for n_osc, color in zip(osc_counts, colors):
    rep_q = [RepetitionCodeUPDE(n_osc, d).physical_qubit_count() for d in distances]
    sc_q = [SurfaceCodeUPDE(n_osc, d).total_qubits for d in distances]
    ax1.plot(distances, rep_q, "o--", color=color, alpha=0.5, label=f"Rep. N={n_osc}")
    ax1.plot(distances, sc_q, "s-", color=color, label=f"Surface N={n_osc}")

ax1.set_xlabel("Code distance d")
ax1.set_ylabel("Physical qubits")
ax1.set_title("Repetition vs Surface Code")
ax1.set_yscale("log")
ax1.legend(fontsize=8, ncol=2)
ax1.grid(True, alpha=0.3)

# Right: circuit depth comparison
for d in [3, 5]:
    depths = []
    for n_osc in range(2, 13, 2):
        sc = SurfaceCodeUPDE(n_osc, d)
        qc = sc.build_step_circuit(dt=0.1)
        depths.append(qc.depth())
    ax2.plot(range(2, 13, 2), depths, "o-", label=f"d={d}")

ax2.set_xlabel("Oscillators")
ax2.set_ylabel("Circuit depth (1 Trotter step)")
ax2.set_title("Surface-Code Step Circuit Depth")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. When Does Surface Code Become Feasible?

Current IBM hardware: 156 qubits (ibm_fez). Future: 1000+ (IBM Condor).

In [ ]:
print("Feasibility on current/future hardware:")
print(f"{'Config':>20} {'Qubits':>8} {'ibm_fez (156q)':>15} {'Condor (1000q)':>15}")
print("-" * 62)
for n_osc, d in [(2, 3), (4, 3), (4, 5), (8, 3), (8, 5), (16, 3), (16, 5)]:
    q = SurfaceCodeUPDE(n_osc, d).total_qubits
    fez = "YES" if q <= 156 else "NO"
    condor = "YES" if q <= 1000 else "NO"
    print(f"{n_osc}osc d={d:>14} {q:8d} {fez:>15} {condor:>15}")